In [18]:
import sys
import os

sys.path.append(os.path.abspath("../"))

import pandas as pd

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory
from utils.snapshot_model import load_validation_results


## Config

**Feature engineering experiment notebook.**

This notebook retrains all models on the existing GCS data snapshot (`v3.4_real`) after
applying a feature engineering change (e.g. log-transforming count/velocity features).
No new data is pulled from BigQuery — the only variable changing between this run and
the v5.1 baseline is the feature engineering logic.

- **Data loaded from GCS:** `v3.4_real` (no BQ call, no data version bump)
- **Hyperparams:** reuses existing snapshot (`v1.1`) — no tuning
- **Model version:** minor bump (`v5.1 → v5.2`) — same data, new feature representation
- **Validation:** evaluated against the locked holdout (`splits/validation_ids.json`)

Set `dry_run = False` only when you're ready to write model artifacts and results to GCS.

In [ ]:
# Set False only when you're ready to write to GCS.
dry_run = True

config = (
    VersionConfig.load(use_synthetic=False)
    .snapshot_models()    # minor version bump (feature engineering, same data)
    .build()
)
run = PipelineRun(config)
stages = PipelineFactory.retrain_existing_data(config)

print('\nScenario:', stages.scenario)
print('data (read from GCS):', config.raw_version)
print('model version (write):', config.next_model_version)
print('dry_run:', dry_run)

VersionConfig loaded:
  data:          v3.4 (raw_suffix='real')
  baselines:     v4.0
  model:         v5.2
  hyperparams:   v1.1
  use_synthetic: False

VersionConfig ready:
  Active flags      : ['models']
  data              : v3.4 (unchanged)
  baselines         : v4.0 (unchanged)
  model             : v5.2 -> v5.3
  hyperparams       : v1.1 (unchanged)
  raw_version       : v3.4_real
  next_final_version: v3.4_100real
  model_version     : v5.2  ->  v5.3
  baselines_version : v4.0
  hyperparam_version: v1.1
  use_synthetic     : False

  Call config.commit() after all snapshots succeed.

Scenario: retrain_existing_data
data (read from GCS): v3.4_real
model version (write): v5.3
dry_run: False


---
## Stage 1 — DataLoader (GCS)

Reads the parquet snapshot at `config.raw_version`. **No BigQuery call.**

In [20]:
stages.loader.run(run)
run.summary()

Loaded snapshot 'v3.4_real': 60696 rows from 2026-04-29
  Polls: {'upload': 21398, '24h': 20906, '7d': 18392}
Loaded baselines 'v4.0': 28814 baseline videos, 974 median rows (974 channels)
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       None
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  df_gen         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  X_gen          None
  y_train        None
  y_test         None
  y_val          None
  y_gen          None
  models         empty dict
  results        empty dict


## Stage 2 — DataPreprocessor

Pivots the long-format poll records into one row per video, joins channel baselines,
and runs structural cleanup. Writes `run.df_clean`.

In [21]:
stages.preprocessor.run(run)
run.summary()

Building clean dataset
snapshot cols: Index(['video_id', 'poll_timestamp', 'channel_id', 'channel_handle', 'title',
       'view_count', 'like_count', 'comment_count', 'face_count', 'brightness',
       'colorfulness', 'vertical', 'tier', 'description', 'tags',
       'duration_seconds', 'category_id', 'category_name', 'published_at',
       'poll_label', 'hours_since_publish', 'subscriber_count',
       'contains_synthetic_media'],
      dtype='object')

[1/3] Pivoting snapshots...
  Videos with all 3 polls: 18334 (dropped 3111 incomplete)
  Pivoted shape: (18403, 34)

[2/3] Joining baseline medians...
  Baseline join: 18403/18403 videos matched a channel median

[3/3] Cleaning data...
  Cleaned: 18403 rows × 40 columns

Clean dataset: 18403 rows × 40 columns
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFr

## Stage 3 — FeatureEngineer

Runs the full feature engineering chain on `run.df_clean`.

**This is where the experiment lives.** The log-transform changes are applied inside
`feature_engineering.py` — count/velocity features are passed through `np.log1p()`
before returning. Rates, ratios, binary flags, and categorical encodings are unchanged.
Writes `run.df_engineered`.

In [22]:
stages.engineer.run(run)
run.summary()

  all: dropped 414 rows with NaN in a baseline_median_* column or 0.0 baseline_median_engagement_rate
Engineering features

[1/10] Computing target variable...
  Target distribution: 55.1% above baseline, 44.9% below

[2/10] Computing velocity features...
  Computed velocity, upload momentum, normalized velocity, and acceleration features

[3/10] Computing ratio and baseline-normalized features...
  Computed ratio and baseline-normalized features

[4/10] Computing subscriber-normalized metrics...
  Computed subscriber-normalized metrics for upload/24h/7d

[5/10] Computing categorical features...
  Title categories:
title_category
neutral        11344
all_caps        2048
exclamation     1955
question        1572
listicle         562
how_to           230
clickbait        204
emoji_heavy       74
  Description categories:
desc_category
link_heavy        4729
minimal           4509
has_links         3837
has_hashtags      2697
neutral           1395
has_timestamps     774
long_form       

## Stage 4 — DataSplitter

Loads locked validation IDs from GCS (`splits/validation_ids.json`).
Filters `run.df_engineered` to produce `df_val`, then stratifies the remaining
rows 80/20 into `df_train` / `df_test`.

In [23]:
stages.splitter.run(run)
run.summary()

DataSplitter — loaded holdout (5,193 val rows):
  df_val:    5,193 rows (28.9%)
  df_train: 10,236 rows (56.9%)
  df_test:   2,560 rows (14.2%)
DataSplitter — no generalization-vertical rows (Music/Sports) found.
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 89)
  df_train       populated  DataFrame shape=(10236, 89)
  df_test        populated  DataFrame shape=(2560, 89)
  df_val         populated  DataFrame shape=(5193, 89)
  df_gen         populated  DataFrame shape=(0, 89)
  X_train        populated  DataFrame shape=(10236, 55)
  X_test         populated  DataFrame shape=(2560, 55)
  X_val          populated  DataFrame shap

## Stage 5 — Scaler

Fits `StandardScaler` on `X_train` (which now receives log-transformed inputs),
transforms `X_train`, `X_test`, and `X_val`. Captures `run.X_val_unscaled` so
Validator can apply each model's own historical scaler.

In [24]:
stages.scaler.run(run)
run.summary()

PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 89)
  df_train       populated  DataFrame shape=(10236, 89)
  df_test        populated  DataFrame shape=(2560, 89)
  df_val         populated  DataFrame shape=(5193, 89)
  df_gen         populated  DataFrame shape=(0, 89)
  X_train        populated  DataFrame shape=(10236, 55)
  X_test         populated  DataFrame shape=(2560, 55)
  X_val          populated  DataFrame shape=(5193, 55)
  X_val_unscaled  populated  DataFrame shape=(5193, 55)
  X_gen          populated  DataFrame shape=(0, 55)
  y_train        populated  Series length=10236
  y_test         populated  Series length=25

## Stage 6 — Trainer

Trains LR, RF, XGB, and VotingClassifier using the existing hyperparams snapshot
(`v1.1`). No hyperparameter search — isolates the feature engineering change as the
sole variable vs. the v5.1 baseline.

In [25]:
stages.trainer.run(run)
run.summary()

Loaded hyperparams 'v1.1' (saved 2026-04-29)
  Models: ['LogisticRegression', 'RandomForest', 'XGBoost', 'VotingClassifier']
  Search: {'strategy': 'random', 'n_iter': 100, 'cv': 5, 'scoring': 'roc_auc'}
Loaded hyperparams from snapshot 'v1.1'.
  Note: injected l1_ratio=0.5 for elasticnet LR (missing from snapshot).
Training LogisticRegression (L1)...
Training RandomForestClassifier...
Training XGBClassifier...
Training VotingClassifier ensemble (RF + XGB, weights=[1, 2])...

=== ModelTrainer — test-set results ===
  lr_l1         AUC=0.7719  acc=0.7117  F1↑=0.7446
  rf            AUC=0.8645  acc=0.7805  F1↑=0.8079
  xgb           AUC=0.9085  acc=0.8262  F1↑=0.8446
  ensemble      AUC=0.9049  acc=0.8203  F1↑=0.8398
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populat

## Stage 7 — ModelSnapshotter

Saves trained model artifacts (model, scaler, feature_cols, metadata) to GCS
under `vMaj.min_*`.

In [26]:
if not dry_run:
    stages.model_snapshotter.run(run)
    print('\nSaved models:', list(run.models.keys()))
else:
    print('[dry_run] skipping ModelSnapshotter')

Model artifacts saved to models/v5.3_lr_l1/
  Uploaded gs://maduros-dolce-capstone-data/models/v5.3_lr_l1/model.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.3_lr_l1/scaler.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.3_lr_l1/feature_cols.json
  Uploaded gs://maduros-dolce-capstone-data/models/v5.3_lr_l1/metadata.json

Model v5.3_lr_l1 (LogisticRegression)
  Data: v3.4_100real (10236 real + 0 synthetic)
  ROC-AUC: 0.7719
  Accuracy: 0.7117
  F1 (above): 0.7446
Model artifacts saved to models/v5.3_rf/
  Uploaded gs://maduros-dolce-capstone-data/models/v5.3_rf/model.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.3_rf/scaler.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.3_rf/feature_cols.json
  Uploaded gs://maduros-dolce-capstone-data/models/v5.3_rf/metadata.json

Model v5.3_rf (RandomForest)
  Data: v3.4_100real (10236 real + 0 synthetic)
  ROC-AUC: 0.8645
  Accuracy: 0.7805
  F1 (above): 0.8079
Model artifacts saved to models/v5.3_xgb/
  

## Stage 8 — Validator

Evaluates each model on the locked validation set using `X_val_unscaled` +
each model's own freshly-fitted scaler. Populates `run.results`.

In [27]:
stages.validator.run(run)
run.summary()


=== Validator — validation-set results ===
  lr_l1           AUC=0.7681  acc=0.7004  F1↑=0.7330
  rf              AUC=0.8718  acc=0.7928  F1↑=0.8204
  xgb             AUC=0.9090  acc=0.8302  F1↑=0.8487
  ensemble        AUC=0.9058  acc=0.8305  F1↑=0.8495
PipelineRun(config=VersionConfig(data=(3, 4), model=(5, 3), baselines=(4, 0), hyperparams=(1, 1), use_synthetic=False, active=['models']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 89)
  df_train       populated  DataFrame shape=(10236, 89)
  df_test        populated  DataFrame shape=(2560, 89)
  df_val         populated  DataFrame shape=(5193, 89)
  df_gen         populated  DataFrame shape=(0, 89)
  X_train        populated  DataFrame shape=(10236, 55)
  X_test         populated  DataFrame shape=(2560, 55)

## Stage 9 — Persist Validation Results to GCS

In [28]:
if not dry_run:
    stages.validation_results_snapshotter.run(run)
    # Appends to gs://maduros-dolce-capstone-data/models/{model_version}/validation_results.jsonl
else:
    print('[dry_run] skipping ValidationResultsSnapshotter')

Validation results appended → gs://maduros-dolce-capstone-data/models/v5.3/validation_results.jsonl
  lr_l1           AUC=0.7681  acc=0.7004  F1↑=0.7330
  rf              AUC=0.8718  acc=0.7928  F1↑=0.8204
  xgb             AUC=0.9090  acc=0.8302  F1↑=0.8487
  ensemble        AUC=0.9058  acc=0.8305  F1↑=0.8495


---
## Results

Compares this run against the previous model version (`config.model_version`) loaded from GCS.

In [29]:
metric_cols = [
    'roc_auc', 'accuracy',
    'f1_above', 'precision_above', 'recall_above',
    'f1_below', 'precision_below', 'recall_below',
]

# Current run results
df_current = (
    pd.DataFrame(run.results).T[metric_cols]
    .astype(float).round(4)
)
df_current.index.name = 'model'

# Baseline: most recent saved run for the previous model version
df_history = load_validation_results(config.model_version)
if not df_history.empty:
    latest_ts = df_history.index.get_level_values('run_timestamp').max()
    df_baseline = (
        df_history.xs(latest_ts, level='run_timestamp')[metric_cols]
        .astype(float).round(4)
    )
    df_baseline.index = df_baseline.index.get_level_values('model_name')
    df_baseline.index.name = 'model'

    df_compare = df_current[['roc_auc']].rename(columns={'roc_auc': f'{config.next_model_version} AUC'})
    df_compare[f'{config.model_version} AUC'] = df_baseline['roc_auc']
    df_compare['delta'] = (df_compare[f'{config.next_model_version} AUC'] - df_compare[f'{config.model_version} AUC']).round(4)
    print(df_compare.to_string())
    print()

print('Full metrics — current run:')
print(df_current.to_string())


Loaded 8 validation result records
          v5.3 AUC  v5.2 AUC   delta
model                               
lr_l1       0.7681    0.7681  0.0000
rf          0.8718    0.8723 -0.0005
xgb         0.9090    0.9104 -0.0014
ensemble    0.9058    0.9068 -0.0010

Full metrics — current run:
          roc_auc  accuracy  f1_above  precision_above  recall_above  f1_below  precision_below  recall_below
model                                                                                                        
lr_l1      0.7681    0.7004    0.7330           0.7255        0.7406    0.6586           0.6674        0.6501
rf         0.8718    0.7928    0.8204           0.7910        0.8519    0.7552           0.7954        0.7189
xgb        0.9090    0.8302    0.8487           0.8400        0.8575    0.8065           0.8173        0.7960
ensemble   0.9058    0.8305    0.8495           0.8383        0.8610    0.8062           0.8203        0.7926


In [30]:
# Top features per model — check whether log-transform shifted LR coefficient distribution
for name, entry in run.results.items():
    top = entry.get('top_features', [])[:5]
    print(f'\n{name}:')
    for f in top:
        key = 'coefficient' if 'coefficient' in f else 'importance'
        print(f'  {f["feature"]:35s}  {f[key]:+.4f}')


lr_l1:
  like_count_upload_vs_baseline        +12.9466
  view_count_upload_vs_baseline        -4.1057
  like_rate_24h                        +1.1597
  view_count_velocity_24h              -0.7925
  like_count_velocity_24h              +0.5288

rf:
  like_rate_24h                        +0.0990
  view_velocity_ratio                  +0.0459
  like_count_upload_vs_baseline        +0.0392
  like_rate_upload                     +0.0378
  view_count_upload_vs_baseline        +0.0311

xgb:
  baseline_baseline_video_count        +0.0654
  like_rate_24h                        +0.0565
  tier_encoded                         +0.0542
  like_count_upload_vs_baseline        +0.0297
  view_count_upload_vs_baseline        +0.0294

ensemble:


---
## Final: GCS Write

Commit model version bump (`vMaj.min → vMaj.min+1`) to GCS. Run only after all snapshots succeed.

In [31]:
if not dry_run:
    config.commit()
else:
    print('[dry_run] skipping config.commit()')


Committed versions.json -> data v3.4, model v5.3, baselines v4.0, hyperparams v1.1
